# Feature importance and partial dependence plots

A court uses an opaque gradient boosting model that predicts whether a defendant is rearrested
within two years. Two questions about the **whole model**:

1. *Feature importance:* how much does the model rely on each feature?
2. *Partial dependence:* how does the prediction change when one feature changes?

The data is synthetic, so we can compare each answer with the true process that made the labels.

In [1]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from recidivism_data import FEATURES, load, load_xy, onehot, true_probability
from recidivism_data.shared import new_gbm, opaque_model, person
from recidivism_data.theme import BLUE, MUTED, ORANGE, use_theme
from sklearn.inspection import PartialDependenceDisplay, permutation_importance

warnings.filterwarnings("ignore")
use_theme()

X_train, y_train = load_xy("train")
X_test, y_test = load_xy("test")
model = opaque_model()
print(f"test accuracy of the opaque model: {model.score(X_test, y_test):.3f}")

test accuracy of the opaque model: 0.675


## 1. Permutation feature importance

**By hand.** Shuffle the values of one feature in the test data. The model then gets a value
from a random other person, so the feature has no information. The decrease of the accuracy
shows how much the model relied on the feature. No retraining is necessary.

In [2]:
rng = np.random.default_rng(0)
baseline = model.score(X_test, y_test)
for feature in ["priors_count", "county"]:
    shuffled = X_test.copy()
    shuffled[feature] = rng.permutation(shuffled[feature].to_numpy())
    accuracy = model.score(shuffled, y_test)
    print(
        f"shuffle {feature:<13} accuracy {baseline:.3f} -> {accuracy:.3f}  (decrease {baseline - accuracy:.3f})"
    )

shuffle priors_count  accuracy 0.675 -> 0.639  (decrease 0.036)
shuffle county        accuracy 0.675 -> 0.676  (decrease -0.000)


**With scikit-learn**, for all features, 30 repeats each (a different shuffle gives a different
number). For comparison, the same measurement for the true process: `age_at_first_arrest` and
`county` are no causes of a rearrest in the data generator.

In [3]:
class TrueProcess:
    """The data generator, in the form of a model."""

    def fit(self, X, y):
        return self

    def predict(self, X):
        return (true_probability(X) > 0.5).astype(int)

    def score(self, X, y):
        return np.mean(self.predict(X) == y)


def importance(estimator):
    result = permutation_importance(estimator, X_test, y_test, n_repeats=30, random_state=0)
    return pd.DataFrame(
        {"mean": result.importances_mean, "std": result.importances_std}, index=FEATURES
    )


imp_model, imp_truth = importance(model), importance(TrueProcess())
order = imp_model["mean"].sort_values().index
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6), sharey=True, sharex=True)
for ax, imp, title, color in [
    (axes[0], imp_model, "The opaque model", BLUE),
    (axes[1], imp_truth, "The true process", ORANGE),
]:
    ax.barh(
        order,
        imp.loc[order, "mean"],
        xerr=imp.loc[order, "std"],
        color=color,
        height=0.6,
        ecolor=MUTED,
    )
    ax.set_title(title)
    ax.set_xlabel("decrease of accuracy when shuffled")
    ax.grid(axis="y", visible=False)
plt.show()
pd.DataFrame({"model": imp_model["mean"], "true process": imp_truth["mean"]}).loc[
    order[::-1]
].round(3)

,model,true process
age,0.031,0.048
priors_count,0.027,0.041
age_at_first_arrest,0.017,0.000
employed,0.013,0.008
charge_type,0.010,0.012
juvenile_priors,0.006,0.007
sex,0.005,0.004
in_treatment,0.004,0.007
charge_degree,0.004,-0.000
county,0.000,0.000


The model relies on `age_at_first_arrest`, which is not a cause. It is correlated with `age`
and `priors_count`, and the model divides the credit between them. Feature importance tells
you what **the model** uses, not what causes the outcome.

**Pitfall: correlated features.** Shuffling says "`age` is important for this model". It does
not say "without `age` the model would be much worse". Retrain without the feature to see that:

In [4]:
for feature in ["age", "age_at_first_arrest", "priors_count", "employed"]:
    keep = [c for c in onehot(X_train).columns if c != feature]
    retrained = new_gbm().fit(onehot(X_train)[keep], y_train)
    drop = baseline - retrained.score(onehot(X_test)[keep], y_test)
    print(
        f"{feature:<20} shuffled: -{imp_model.loc[feature, 'mean']:.3f}   retrained without it: {-drop:+.3f}"
    )

age                  shuffled: -0.031   retrained without it: -0.014
age_at_first_arrest  shuffled: -0.017   retrained without it: +0.005


priors_count         shuffled: -0.027   retrained without it: -0.009
employed             shuffled: -0.013   retrained without it: -0.004


## 2. Partial dependence plot (PDP)

**By hand, for one person.** Take the 24-year-old man that the other projects explain, change
only his age, and ask the model each time.

In [5]:
ages = np.arange(18, 66)


def curve(row: pd.DataFrame, predict) -> np.ndarray:
    varied = pd.concat([row] * len(ages), ignore_index=True)
    varied["age"] = ages
    return predict(varied)


def predict_model(X):
    return model.predict_proba(X)[:, 1]


one = person()
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(ages, curve(one, predict_model))
ax.scatter(one["age"], predict_model(one), color=BLUE, zorder=3, s=40)
ax.set(
    xlabel="age",
    ylabel="predicted P(rearrest)",
    title="One person: the prediction when only the age changes",
    ylim=(0, 1),
)
plt.show()

**For many people.** Each thin line is one person (an *individual conditional expectation*
curve). **Their average is the partial dependence plot.** The dashed line is the same average
for the true process.

In [6]:
sample = X_test.sample(300, random_state=0)
ice = np.array([curve(sample.iloc[[i]], predict_model) for i in range(len(sample))])
truth = np.array([curve(sample.iloc[[i]], true_probability) for i in range(len(sample))])

fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(ages, ice[:60].T, color=BLUE, alpha=0.15, linewidth=1)
ax.plot(
    ages, ice.mean(axis=0), color=BLUE, linewidth=3, label="average = partial dependence (model)"
)
ax.plot(
    ages,
    truth.mean(axis=0),
    color=ORANGE,
    linestyle="--",
    label="the same average for the true process",
)
ax.set(
    xlabel="age",
    ylabel="predicted P(rearrest)",
    title="60 individual curves and the partial dependence",
    ylim=(0, 1),
)
ax.legend()
plt.show()

The shape is easy to read: the risk decreases quickly between 18 and about 30, then slowly.
But the model's curve is flatter than the truth, because the model gives a part of the age
effect to `age_at_first_arrest`.

**scikit-learn** makes the plot for each feature. The plot assumes that you can change one
feature and keep the others.

In [7]:
numeric = [
    "age",
    "priors_count",
    "juvenile_priors",
    "age_at_first_arrest",
    "employed",
    "in_treatment",
]
X_float = X_test.astype({c: float for c in numeric})
fig, axes = plt.subplots(2, 3, figsize=(10, 5))
PartialDependenceDisplay.from_estimator(model, X_float, numeric, ax=axes, line_kw={"color": BLUE})
for ax in axes.ravel():
    ax.set_ylim(0.15, 0.65)
fig.suptitle("Partial dependence of the opaque model", x=0.01, ha="left", fontweight="bold")
plt.show()

## 3. Interactions: partial dependence for two features

In the data generator, treatment decreases the risk **only for people with a drug charge**.
A plot for one feature shows only the average effect of `in_treatment`. A partial dependence
for two features shows the interaction.

In [8]:
rows = []
for treatment in [0, 1]:
    for charge in ["drug", "property", "violent", "traffic", "other"]:
        changed = sample.assign(in_treatment=treatment, charge_type=charge)
        rows.append(
            (charge, treatment, predict_model(changed).mean(), true_probability(changed).mean())
        )
grid = pd.DataFrame(rows, columns=["charge_type", "in_treatment", "model", "truth"])
table = grid.pivot(index="charge_type", columns="in_treatment", values=["model", "truth"]).round(2)
table[("model", "effect")] = table[("model", 1)] - table[("model", 0)]
table[("truth", "effect")] = table[("truth", 1)] - table[("truth", 0)]
table.sort_index(axis=1)

model              truth             
in_treatment     0     1 effect     0     1 effect
charge_type                                       
drug          0.45  0.31  -0.14  0.46  0.28  -0.18
other         0.41  0.35  -0.06  0.40  0.40   0.00
property      0.43  0.42  -0.01  0.44  0.44   0.00
traffic       0.31  0.30  -0.01  0.30  0.30   0.00
violent       0.40  0.35  -0.05  0.38  0.38   0.00

In [9]:
fig, ax = plt.subplots(figsize=(6, 4.2))
PartialDependenceDisplay.from_estimator(
    model,
    X_float,
    [("age", "priors_count")],
    ax=ax,
    grid_resolution=30,
    contour_kw={"cmap": "Blues"},
)
fig.suptitle("Partial dependence on age and priors together", x=0.01, ha="left", fontweight="bold")
plt.show()

## 4. Pitfall: the plot asks the model about impossible people

To compute the curve at age 20, **each** person gets the age 20, also a person who was first
arrested at 35 and a person with 15 priors. The model never saw such people in training, and
its answers for them go into the average.

In [10]:
df = load()
print(
    f"largest number of priors of a real {20}-year-old: {df.loc[df['age'] <= 20, 'priors_count'].max()}"
)
for age in [20, 30, 45]:
    impossible = (sample["age_at_first_arrest"] > age).mean()
    print(f"curve at age {age}: {impossible:.0%} of the people have a first arrest after that age")

largest number of priors of a real 20-year-old: 13
curve at age 20: 50% of the people have a first arrest after that age
curve at age 30: 23% of the people have a first arrest after that age
curve at age 45: 6% of the people have a first arrest after that age
